# Gated Experiment: Diagnostic-Gated Aggregation

This notebook runs the expert-weighted method **with prediction-only failure
gating**. For each class, the diagnostic signals decide whether expert
identification looks unreliable; gated classes fall back to a capped
full-ensemble vote instead of trusting the expert subset.

All thresholds and the fallback budget are exposed as knobs. The defaults match
the values reported in the paper (Section 7.4); tighten them for stricter
detection or loosen them for looser detection and observe the effect on the
gated macro-F1.

Like the main experiment, this runs entirely off the prediction JSONs — no raw
datasets are needed.


In [ ]:
import sys, os, json, glob
sys.path.append(os.path.join(os.getcwd(), "src"))

import numpy as np

from io_utils import load_dataset_predictions, make_test_sizes
from pipeline import build_subsample_indices, build_prediction_sets
from dunn import find_dunn_experts
from voting import experts_weighted_voting, predictions_to_vector
from signals import separation_gap, over_bias
from disagreement import class_disagreement_matrix
from gating import gate_class, gated_predictions
from metrics import summarize_method_performance

## Configuration

Paths, protocol constants, and the **gating knobs**. The thresholds default to
the paper's Section 7.4 values; `DELTAS` sweeps the fallback budget for gated
classes (0 = ignore the gated class, 1 = full-ensemble vote comparable to a
typical non-gated class).

In [ ]:
# --- paths ---
PREDICTIONS_DIR = "data/predictions"
OUTPUT_JSONL    = "data/results/gated_records.jsonl"

# --- protocol constants ---
NUM_SIZES         = 20
TRIALS_PER_SIZE   = 3
MIN_SUBSET_SIZE   = 3
MAX_SUBSET_SIZE   = 10
ENFORCE_NUM_MODELS = 15

# --- gating knobs (defaults = paper Section 7.4) ---
TH_CENTRALITY_DISP = 0.035   # A1: gate if centrality dispersion < this
TH_SEP_GAP         = 0.535   # A2 separation-gap precondition
TH_UNDERBIAS       = 0.8     # A2-under threshold
TH_OVERBIAS        = 2.0     # A2-over threshold
DELTAS             = [0.7, 0.8, 0.9, 1.0]   # fallback budget sweep; 1.0 reproduces the paper

os.makedirs(os.path.dirname(OUTPUT_JSONL), exist_ok=True)

## Gated run for one subsample

For each subsample we identify experts, compute the per-class separation gap and
over-bias, ask the gate whether to distrust each class, and then form gated
predictions over the delta sweep. We record the ungated expert-weighted macro-F1
alongside the gated macro-F1 at each delta, so the rescue effect is directly
visible.

In [ ]:
def run_gated_subsample(subs_idx, predictions, y_true, num_classes, seed):
    import random
    random.seed(seed); np.random.seed(seed)

    size = len(subs_idx)
    model_pool = list(predictions.keys())
    subs_true = y_true[subs_idx]

    pred_sets = build_prediction_sets(subs_idx, model_pool, predictions, num_classes)
    expert_partition, _, _ = find_dunn_experts(
        pred_sets, min_subset_size=MIN_SUBSET_SIZE, max_subset_size=MAX_SUBSET_SIZE)
    expW_cls, expW_weights = experts_weighted_voting(expert_partition, pred_sets)

    # ungated expert-weighted baseline
    y_ungated = predictions_to_vector(expW_cls, size)
    ungated_f1 = summarize_method_performance(subs_true, y_ungated, num_classes)["macro_f1"]

    # per-class gate decisions
    gate_flags, fired_map = {}, {}
    for c in range(num_classes):
        sets_c = pred_sets.get(c, {}) or {}
        ids = sorted(sets_c.keys())
        local = {g: i for i, g in enumerate(ids)}
        D = class_disagreement_matrix(sets_c)
        el = [local[g] for g in expert_partition.get(c, ()) if g in local]
        sg = separation_gap(D, el)
        ob = over_bias(pred_sets, expert_partition.get(c, ()), c, model_pool, num_classes)
        gated, fired = gate_class(
            c, pred_sets, expert_partition, model_pool, size,
            sep_gap=(sg if sg == sg else 0.0), over_bias_value=(ob if ob == ob else 0.0),
            th_centrality_disp=TH_CENTRALITY_DISP, th_sep_gap=TH_SEP_GAP,
            th_underbias=TH_UNDERBIAS, th_overbias=TH_OVERBIAS)
        gate_flags[c] = gated
        if gated:
            fired_map[c] = fired

    # gated predictions over the delta sweep
    gated_y = gated_predictions(
        pred_sets, expert_partition, expW_weights, gate_flags,
        num_classes, size, model_pool, deltas=DELTAS)
    gated_f1 = {
        float(d): summarize_method_performance(subs_true, y, num_classes)["macro_f1"]
        for d, y in gated_y.items()
    }

    return {
        "size": int(size), "seed": int(seed),
        "ungated_macro_f1": float(ungated_f1),
        "gated_macro_f1": gated_f1,
        "gated_classes": {int(c): fired_map[c] for c in fired_map},
        "num_gated": int(sum(1 for v in gate_flags.values() if v)),
    }

## Run over all datasets

Sweep sizes and trials for each dataset and write gated records to JSONL.

In [ ]:
json_paths = sorted(glob.glob(os.path.join(PREDICTIONS_DIR, "*_test_predictions_logits.json")))
print(f"Found {len(json_paths)} prediction file(s)")
open(OUTPUT_JSONL, "w").close()

for path in json_paths:
    loaded = load_dataset_predictions(path, enforce_num_models=ENFORCE_NUM_MODELS)
    sizes = make_test_sizes(loaded["n_test_min"], len(loaded["y_true"]), NUM_SIZES)
    n_total = len(loaded["y_true"])

    n_written = 0
    with open(OUTPUT_JSONL, "a") as f:
        for size in sizes:
            for trial in range(TRIALS_PER_SIZE):
                seed = int(size * 100 + trial)
                subs_idx = build_subsample_indices(n_total, size, seed)
                rec = run_gated_subsample(
                    subs_idx, loaded["predictions"], loaded["y_true"],
                    loaded["num_classes"], seed)
                rec["dataset"] = loaded["dataset"]; rec["trial"] = int(trial)
                f.write(json.dumps(rec) + "\n")
                n_written += 1
    print(f"  {loaded['dataset']}: {n_written} gated records")

print("Done.")

## Rescue summary

For each dataset, compare mean ungated macro-F1 against mean gated macro-F1 at
each delta. Datasets with a genuine A2 failure (e.g. a collapsing minority
class) should show a clear lift under gating; healthy datasets should be largely
unchanged.

In [ ]:
import collections
rows = collections.defaultdict(lambda: {"ungated": [], "gated": collections.defaultdict(list)})
with open(OUTPUT_JSONL) as f:
    for line in f:
        r = json.loads(line)
        rows[r["dataset"]]["ungated"].append(r["ungated_macro_f1"])
        for d, v in r["gated_macro_f1"].items():
            rows[r["dataset"]]["gated"][float(d)].append(v)

for ds, data in rows.items():
    ung = sum(data["ungated"]) / len(data["ungated"])
    print(f"\n{ds}: ungated mean macro-F1 = {ung:.4f}")
    for d in sorted(data["gated"]):
        vals = data["gated"][d]
        print(f"   delta={d:.2f}: gated mean macro-F1 = {sum(vals)/len(vals):.4f}")